<a href="https://colab.research.google.com/github/boehnen/satlens/blob/main/notebooks/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# satlens — training

Fine-tunes SegFormer-b0 on OpenEarthMap. Run top to bottom on a T4 GPU (`Runtime → Change runtime type → T4 GPU`). Takes ~45 min.


## 1. Install dependencies

In [1]:
!pip install -q transformers datasets torch torchvision huggingface_hub Pillow tqdm matplotlib

## 2. Imports

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    get_scheduler,
)
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

Using device: cuda


## 3. Download OpenEarthMap dataset

In [3]:
# dataset: OpenEarthMap — https://zenodo.org/records/7223446
# downloaded directly via zenodo API download link
import zipfile, glob

!wget -q --show-progress "https://zenodo.org/records/7223446/files/OpenEarthMap.zip?download=1" -O /content/oem.zip

with zipfile.ZipFile('/content/oem.zip', 'r') as z:
    z.extractall('/content/oem')

imgs  = glob.glob('/content/oem/**/images/*.tif', recursive=True)
masks = glob.glob('/content/oem/**/labels/*.tif', recursive=True)
print(f"images: {len(imgs)}, masks: {len(masks)}")

/content/oem.zip    100%[===================>]   8.47G  13.5MB/s    in 10m 48s 
images: 3838, masks: 3500


## 4. Explore the data

In [4]:
CLASSES = [
    'Background',
    'Bareland',
    'Rangeland',
    'Developed space',
    'Road',
    'Tree',
    'Water',
    'Agriculture land',
    'Building',
]

PALETTE = np.array([
    [0,   0,   0],
    [128, 0,   0],
    [0,   128, 0],
    [128, 128, 0],
    [255, 255, 0],
    [0,   64,  0],
    [0,   0,   255],
    [0,   255, 128],
    [255, 0,   0],
], dtype=np.uint8)

NUM_CLASSES = len(CLASSES)
print(f"{NUM_CLASSES} classes")

9 classes


## 5. Dataset class

In [5]:
import os
import glob
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader

processor = SegformerImageProcessor(
    do_resize=True,
    size={'height': 512, 'width': 512},
    do_normalize=True,
)

# build matched pairs only
img_paths  = sorted(glob.glob('/content/oem/**/images/*.tif', recursive=True))
mask_paths = sorted(glob.glob('/content/oem/**/labels/*.tif', recursive=True))

# index masks by filename so we can match them to images
mask_index = {os.path.basename(p): p for p in mask_paths}
pairs = [
    (img, mask_index[os.path.basename(img)])
    for img in img_paths
    if os.path.basename(img) in mask_index
]

print(f"matched pairs: {len(pairs)}")

# 90/10 train/val split
split = int(len(pairs) * 0.9)
train_pairs = pairs[:split]
val_pairs   = pairs[split:]

class OEMDataset(Dataset):
    def __init__(self, pairs, processor, augment=False):
        self.pairs     = pairs
        self.processor = processor
        self.augment   = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = Image.open(img_path).convert('RGB')
        mask  = Image.open(mask_path)

        if self.augment:
            if torch.rand(1) > 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                mask  = mask.transpose(Image.FLIP_LEFT_RIGHT)
            if torch.rand(1) > 0.5:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)
                mask  = mask.transpose(Image.FLIP_TOP_BOTTOM)

        encoded = self.processor(
            images=image,
            segmentation_maps=mask,
            return_tensors='pt',
        )
        return {
            'pixel_values': encoded['pixel_values'].squeeze(0),
            'labels':       encoded['labels'].squeeze(0),
        }

train_dataset = OEMDataset(train_pairs, processor, augment=True)
val_dataset   = OEMDataset(val_pairs,   processor, augment=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"{len(train_loader)} train / {len(val_loader)} val batches")

matched pairs: 2687
303 train / 34 val batches


## 6. Load pretrained SegFormer-b0

In [10]:
HF_REPO = 'boehnen/satlens-segformer'

id2label = {i: c for i, c in enumerate(CLASSES)}
label2id = {c: i for i, c in id2label.items()}

try:
    # resume from best saved model
    model = SegformerForSemanticSegmentation.from_pretrained(
        HF_REPO,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    print(f'Resumed from {HF_REPO}')
except:
    # fall back to pretrained if no checkpoint exists yet
    model = SegformerForSemanticSegmentation.from_pretrained(
        'nvidia/mit-b0',
        num_labels=NUM_CLASSES,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    print('Starting from nvidia/mit-b0')

model = model.to(DEVICE)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Resumed from boehnen/satlens-segformer


## 7. Training setup

In [11]:
NUM_EPOCHS    = 20
LR            = 6e-5
WARMUP_STEPS  = 50

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler   = get_scheduler(
    'cosine',
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
)

def mean_iou(preds, labels, num_classes, ignore_index=0):
    """Compute mean Intersection-over-Union across classes."""
    ious = []
    preds  = preds.flatten()
    labels = labels.flatten()
    for cls in range(1, num_classes):  # skip background
        pred_cls  = preds  == cls
        label_cls = labels == cls
        intersection = (pred_cls & label_cls).sum().item()
        union        = (pred_cls | label_cls).sum().item()
        if union > 0:
            ious.append(intersection / union)
    return np.mean(ious) if ious else 0.0

print(f'Training for {NUM_EPOCHS} epochs ({total_steps} total steps)')

Training for 20 epochs (6060 total steps)


## 8. Train

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

best_miou       = 0.0
train_losses    = []
val_mious       = []

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]')

    for batch in pbar:
        pixel_values = batch['pixel_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [val]'):
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels       = batch['labels']

            outputs = model(pixel_values=pixel_values)
            logits  = outputs.logits  # (B, C, H/4, W/4)

            upsampled = torch.nn.functional.interpolate(
                logits, size=labels.shape[-2:], mode='bilinear', align_corners=False
            )
            preds = upsampled.argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)

    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    miou       = mean_iou(all_preds, all_labels, NUM_CLASSES)
    val_mious.append(miou)

    print(f'Epoch {epoch+1} | loss: {avg_loss:.4f} | val mIoU: {miou:.4f}')

    if miou > best_miou:
        best_miou = miou
        model.save_pretrained('/content/satlens-segformer-best')
        processor.save_pretrained('/content/satlens-segformer-best')
        model.push_to_hub('boehnen/satlens-segformer')
        processor.push_to_hub('boehnen/satlens-segformer')
        print(f'  ✅ New best model saved (mIoU: {best_miou:.4f})')

print(f'\nTraining complete. Best mIoU: {best_miou:.4f}')

Epoch 1/20 [train]:   0%|          | 0/303 [00:00<?, ?it/s]

## 9. Plot training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, marker='o')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(val_mious, marker='o', color='green')
ax2.set_title('Validation mIoU')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('mIoU')
ax2.grid(True)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print(f'Best validation mIoU: {best_miou:.4f}')

## 10. Visualize predictions

In [ ]:
model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle('Predictions on validation set', fontsize=14)

for i, ax_row in enumerate(axes):
    sample       = dataset['val'][i]
    image        = sample['image'].convert('RGB')
    true_mask    = np.array(sample['label'])

    encoded = processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = model(**encoded)
    logits = outputs.logits
    upsampled = torch.nn.functional.interpolate(
        logits, size=(512, 512), mode='bilinear', align_corners=False
    )
    pred_mask = upsampled.argmax(dim=1).squeeze(0).cpu().numpy()

    ax_row[0].imshow(image.resize((512, 512)))
    ax_row[0].set_title('Image')
    ax_row[0].axis('off')

    ax_row[1].imshow(PALETTE[true_mask])
    ax_row[1].set_title('Ground truth')
    ax_row[1].axis('off')

    ax_row[2].imshow(PALETTE[pred_mask])
    ax_row[2].set_title('Predicted')
    ax_row[2].axis('off')

patches = [mpatches.Patch(color=PALETTE[i]/255, label=CLASSES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig('/content/predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Upload model to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, login

HF_USERNAME  = 'boehnen'
REPO_NAME    = 'satlens-segformer'
REPO_ID      = f'{HF_USERNAME}/{REPO_NAME}'

login()

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)

model.push_to_hub(REPO_ID)
processor.push_to_hub(REPO_ID)

print(f'\n✅ Model uploaded to: https://huggingface.co/{REPO_ID}')
print(f'\nSet MODEL_ID={REPO_ID} in your Hugging Face Space environment variables.')